<div style="background:linear-gradient(135deg,#081C2C 0%,#0E4D64 58%,#137C8B 100%);padding:38px 42px;border-radius:20px;color:white;box-shadow:0 12px 28px rgba(8,28,44,.22)">
  <div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;color:#B9E3E8">Projeto end-to-end • Ciência de dados aplicada</div>
  <h1 style="font-size:38px;line-height:1.12;margin:12px 0 8px">Risco de crédito<br>da evidência à decisão</h1>
  <p style="font-size:18px;max-width:820px;color:#E6F4F6;margin:0">Uma análise do Home Credit Default Risk que conecta histórico financeiro, comportamento de pagamento e modelagem preditiva — com rigor técnico, narrativa executiva e código simples.</p>
  <div style="margin-top:24px;font-size:13px;color:#B9E3E8">Google Colab • Python • PT-BR • versão reproduzível</div>
</div>

> **Pergunta central:** como reconhecer sinais de dificuldade de pagamento antes da decisão de crédito, sem transformar o modelo em uma caixa-preta?


## Roteiro da apresentação

| Capítulo | Pergunta que vamos responder | Entrega |
|---|---|---|
| 1. Contexto e dados | O que existe e como as tabelas se conectam? | Inventário, qualidade e mapa relacional |
| 2. Retrato do risco | Quem inadimple e quais sinais aparecem primeiro? | Análise exploratória orientada ao negócio |
| 3. Memória financeira | O histórico acrescenta informação útil? | Engenharia de atributos de todas as bases |
| 4. Modelo | Conseguimos ordenar clientes por risco? | Baseline interpretável + modelo não linear |
| 5. Decisão | Qual limiar traduz o score em uma política? | Curvas, custos, decis e matriz de confusão |
| 6. Confiança | O resultado é explicável e governável? | Importância, auditoria por grupos e artefatos |

Cada capítulo termina com um **portão de qualidade**. Se a resposta a “dá para melhorar?” for “sim”, a execução para; só avançamos quando a resposta for “não”.


## 0. Preparação do ambiente

No Colab, a forma mais estável é colocar `homecredit.zip` no Google Drive. Altere apenas as variáveis da próxima célula. O notebook também aceita uma pasta já descompactada por meio da variável de ambiente `HOME_CREDIT_DATA_DIR`.

O modo **completo** usa todos os registros. O modo **rápido** reduz apenas a amostra de modelagem; as agregações históricas continuam corretas.


In [ ]:
# Trazemos de `pathlib` as ferramentas que sustentam esta parte da análise.
from pathlib import Path
# Convidamos `json` para participar deste capítulo do processamento.
import json
# Convidamos `os` para participar deste capítulo do processamento.
import os
# Convidamos `sys` para participar deste capítulo do processamento.
import sys
# Convidamos `time` para participar deste capítulo do processamento.
import time
# Convidamos `warnings` para participar deste capítulo do processamento.
import warnings
# Convidamos `zipfile` para participar deste capítulo do processamento.
import zipfile

# Fixamos `EM_COLAB` como uma decisão explícita e fácil de ajustar pelo leitor.
EM_COLAB = "google.colab" in sys.modules
# Fixamos `USAR_GOOGLE_DRIVE` como uma decisão explícita e fácil de ajustar pelo leitor.
USAR_GOOGLE_DRIVE = False  # Troque para True no Colab se o ZIP estiver no Drive.

# Fixamos `CAMINHO_ZIP` como uma decisão explícita e fácil de ajustar pelo leitor.
CAMINHO_ZIP = Path(os.getenv(
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "HOME_CREDIT_ZIP",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "/content/drive/MyDrive/homecredit.zip" if EM_COLAB else "homecredit.zip",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))
# Fixamos `DIRETORIO_DADOS` como uma decisão explícita e fácil de ajustar pelo leitor.
DIRETORIO_DADOS = Path(os.getenv(
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "HOME_CREDIT_DATA_DIR",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "/content/home_credit_dados" if EM_COLAB else "dados/raw",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))
# Fixamos `PASTA_RESULTADOS` como uma decisão explícita e fácil de ajustar pelo leitor.
PASTA_RESULTADOS = Path(os.getenv(
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "HOME_CREDIT_OUTPUT_DIR",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "/content/resultados_risco_credito" if EM_COLAB else "resultados",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))

# Fixamos `MODO_EXECUCAO` como uma decisão explícita e fácil de ajustar pelo leitor.
MODO_EXECUCAO = os.getenv("HOME_CREDIT_MODE", "completo")  # completo ou rapido
# Fixamos `EXIGIR_BASE_COMPLETA` como uma decisão explícita e fácil de ajustar pelo leitor.
EXIGIR_BASE_COMPLETA = os.getenv("HOME_CREDIT_STRICT", "1") == "1"
# Fixamos `RANDOM_STATE` como uma decisão explícita e fácil de ajustar pelo leitor.
RANDOM_STATE = 42
# Fixamos `CUSTO_FALSO_POSITIVO` como uma decisão explícita e fácil de ajustar pelo leitor.
CUSTO_FALSO_POSITIVO = 1
# Fixamos `CUSTO_FALSO_NEGATIVO` como uma decisão explícita e fácil de ajustar pelo leitor.
CUSTO_FALSO_NEGATIVO = 5  # Hipótese ilustrativa; deve ser calibrada pela instituição.

# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if EM_COLAB and USAR_GOOGLE_DRIVE:
    # Trazemos de `google.colab` as ferramentas que sustentam esta parte da análise.
    from google.colab import drive
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    drive.mount("/content/drive")

# Contamos ao leitor o estado atual da execução para que nenhum passo aconteça no escuro.
print(f"Ambiente: {'Google Colab' if EM_COLAB else 'local'}")
# Contamos ao leitor o estado atual da execução para que nenhum passo aconteça no escuro.
print(f"Modo: {MODO_EXECUCAO} | Exigir base completa: {EXIGIR_BASE_COMPLETA}")


In [ ]:
# Convidamos `joblib` para participar deste capítulo do processamento.
import joblib
# Convidamos `matplotlib.pyplot` para participar deste capítulo do processamento.
import matplotlib.pyplot as plt
# Convidamos `numpy` para participar deste capítulo do processamento.
import numpy as np
# Convidamos `pandas` para participar deste capítulo do processamento.
import pandas as pd
# Convidamos `seaborn` para participar deste capítulo do processamento.
import seaborn as sns

# Trazemos de `IPython.display` as ferramentas que sustentam esta parte da análise.
from IPython.display import HTML, Markdown, display
# Trazemos de `sklearn.base` as ferramentas que sustentam esta parte da análise.
from sklearn.base import clone
# Trazemos de `sklearn.calibration` as ferramentas que sustentam esta parte da análise.
from sklearn.calibration import calibration_curve
# Trazemos de `sklearn.compose` as ferramentas que sustentam esta parte da análise.
from sklearn.compose import ColumnTransformer
# Trazemos de `sklearn.ensemble` as ferramentas que sustentam esta parte da análise.
from sklearn.ensemble import HistGradientBoostingClassifier
# Trazemos de `sklearn.impute` as ferramentas que sustentam esta parte da análise.
from sklearn.impute import SimpleImputer
# Trazemos de `sklearn.inspection` as ferramentas que sustentam esta parte da análise.
from sklearn.inspection import permutation_importance
# Trazemos de `sklearn.linear_model` as ferramentas que sustentam esta parte da análise.
from sklearn.linear_model import LogisticRegression
# Trazemos de `sklearn.metrics` as ferramentas que sustentam esta parte da análise.
from sklearn.metrics import (
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    average_precision_score,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    brier_score_loss,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    confusion_matrix,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f1_score,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    precision_recall_curve,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    precision_score,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    recall_score,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    roc_auc_score,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    roc_curve,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Trazemos de `sklearn.model_selection` as ferramentas que sustentam esta parte da análise.
from sklearn.model_selection import train_test_split
# Trazemos de `sklearn.pipeline` as ferramentas que sustentam esta parte da análise.
from sklearn.pipeline import Pipeline
# Trazemos de `sklearn.preprocessing` as ferramentas que sustentam esta parte da análise.
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
warnings.filterwarnings("ignore", category=FutureWarning)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
pd.set_option("display.max_columns", 140)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Fixamos `CORES` como uma decisão explícita e fácil de ajustar pelo leitor.
CORES = {
    # Registramos `marinho` para manter esta informação nomeada e auditável.
    "marinho": "#0B1F33",
    # Registramos `azul` para manter esta informação nomeada e auditável.
    "azul": "#0E4D64",
    # Registramos `turquesa` para manter esta informação nomeada e auditável.
    "turquesa": "#137C8B",
    # Registramos `coral` para manter esta informação nomeada e auditável.
    "coral": "#FF6B5E",
    # Registramos `dourado` para manter esta informação nomeada e auditável.
    "dourado": "#E6A23C",
    # Registramos `cinza` para manter esta informação nomeada e auditável.
    "cinza": "#667785",
    # Registramos `claro` para manter esta informação nomeada e auditável.
    "claro": "#EAF3F5",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
}
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.set_theme(style="whitegrid", context="notebook")
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.rcParams.update({
    # Registramos `figure.figsize` para manter esta informação nomeada e auditável.
    "figure.figsize": (10, 5),
    # Registramos `axes.titleweight` para manter esta informação nomeada e auditável.
    "axes.titleweight": "bold",
    # Registramos `axes.titlesize` para manter esta informação nomeada e auditável.
    "axes.titlesize": 14,
    # Registramos `axes.labelcolor` para manter esta informação nomeada e auditável.
    "axes.labelcolor": CORES["marinho"],
    # Registramos `axes.edgecolor` para manter esta informação nomeada e auditável.
    "axes.edgecolor": "#D7E2E6",
    # Registramos `text.color` para manter esta informação nomeada e auditável.
    "text.color": CORES["marinho"],
    # Registramos `font.family` para manter esta informação nomeada e auditável.
    "font.family": "DejaVu Sans",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})

# Criamos a função `br_numero` para transformar esta ideia em uma etapa reutilizável.
def br_numero(valor, casas=0):
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return f"{valor:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

# Criamos a função `portao_qualidade` para transformar esta ideia em uma etapa reutilizável.
def portao_qualidade(etapa, criterios):
    # Guardamos em `falhas` a evidência produzida por esta operação.
    falhas = [descricao for descricao, passou in criterios.items() if not bool(passou)]
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if falhas:
        # Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
        display(HTML(
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            f"<div style='border-left:5px solid {CORES['coral']};padding:14px 18px;background:#FFF3F1'>"
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            f"<b>{etapa} — Dá para melhorar? Sim.</b><br>" + "<br>".join(f"• {x}" for x in falhas) + "</div>"
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        ))
        # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
        raise AssertionError("O portão de qualidade encontrou pendências. Corrija antes de avançar.")
    # Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
    display(HTML(
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        f"<div style='border-left:5px solid {CORES['turquesa']};padding:14px 18px;background:#EEF8F7'>"
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        f"<b>{etapa} — Dá para melhorar? Não.</b><br>Todos os critérios definidos para esta etapa foram atendidos.</div>"
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ))


## 1. A matéria-prima da decisão

O Home Credit não descreve apenas uma proposta. Ele registra uma trajetória: a solicitação atual, os créditos vistos pelo bureau, empréstimos anteriores na própria instituição e o comportamento mensal de pagamentos. A unidade final da decisão é o **cliente (`SK_ID_CURR`)**; todas as tabelas históricas serão resumidas para essa granularidade antes da modelagem.


In [ ]:
# Fixamos `ARQUIVOS_ESPERADOS` como uma decisão explícita e fácil de ajustar pelo leitor.
ARQUIVOS_ESPERADOS = [
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "HomeCredit_columns_description.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "application_train.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "application_test.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "bureau.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "bureau_balance.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "previous_application.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "POS_CASH_balance.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "credit_card_balance.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "installments_payments.csv",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "sample_submission.csv",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
]

# Criamos a função `extrair_zip_com_seguranca` para transformar esta ideia em uma etapa reutilizável.
def extrair_zip_com_seguranca(caminho_zip, destino):
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if not caminho_zip.exists():
        # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
        raise FileNotFoundError(
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            f"ZIP não encontrado em {caminho_zip}. Ajuste CAMINHO_ZIP ou HOME_CREDIT_DATA_DIR."
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        )
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    destino.mkdir(parents=True, exist_ok=True)
    # Tentamos executar a operação principal, preparados para explicar uma eventual falha.
    try:
        # Abrimos este recurso de forma controlada para garantir que ele seja encerrado corretamente.
        with zipfile.ZipFile(caminho_zip) as arquivo:
            # Guardamos em `corrompido` a evidência produzida por esta operação.
            corrompido = arquivo.testzip()
            # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
            if corrompido:
                # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
                raise zipfile.BadZipFile(f"Entrada corrompida: {corrompido}")
            # Guardamos em `raiz` a evidência produzida por esta operação.
            raiz = destino.resolve()
            # Percorremos cada elemento para aplicar a mesma regra de forma consistente.
            for membro in arquivo.infolist():
                # Guardamos em `alvo` a evidência produzida por esta operação.
                alvo = (destino / membro.filename).resolve()
                # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
                if alvo != raiz and raiz not in alvo.parents:
                    # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
                    raise ValueError(f"Caminho inseguro dentro do ZIP: {membro.filename}")
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            arquivo.extractall(destino)
    # Traduzimos a falha técnica em uma mensagem clara e acionável para quem executa o projeto.
    except zipfile.BadZipFile as erro:
        # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
        raise zipfile.BadZipFile(
            # Incluímos este elemento na coleção que organiza os componentes da etapa.
            "O arquivo ZIP está incompleto ou corrompido. Baixe-o novamente no Kaggle antes de executar."
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ) from erro

# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if not (DIRETORIO_DADOS / "application_train.csv").exists():
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    extrair_zip_com_seguranca(CAMINHO_ZIP, DIRETORIO_DADOS)

# Fixamos `ARQUIVOS` como uma decisão explícita e fácil de ajustar pelo leitor.
ARQUIVOS = {p.name: p for p in DIRETORIO_DADOS.rglob("*.csv")}
# Guardamos em `faltantes` a evidência produzida por esta operação.
faltantes = [nome for nome in ARQUIVOS_ESPERADOS if nome not in ARQUIVOS]

# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if faltantes:
    # Guardamos em `mensagem` a evidência produzida por esta operação.
    mensagem = "Arquivos ausentes: " + ", ".join(faltantes)
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if EXIGIR_BASE_COMPLETA:
        # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
        raise FileNotFoundError(mensagem + ". Use um download completo ou desative o modo estrito conscientemente.")
    # Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
    display(HTML(f"<div style='background:#FFF8E8;padding:14px;border-left:5px solid {CORES['dourado']}'><b>Modo tolerante:</b> {mensagem}. O notebook seguirá sem inventar dados e registrará a limitação.</div>"))

# Contamos ao leitor o estado atual da execução para que nenhum passo aconteça no escuro.
print(f"{len(ARQUIVOS)} CSV(s) localizado(s) em {DIRETORIO_DADOS}")


In [ ]:
# Criamos a função `contar_linhas_csv` para transformar esta ideia em uma etapa reutilizável.
def contar_linhas_csv(caminho, tamanho_bloco=8 * 1024 * 1024):
    # Guardamos em `linhas` a evidência produzida por esta operação.
    linhas = 0
    # Abrimos este recurso de forma controlada para garantir que ele seja encerrado corretamente.
    with caminho.open("rb") as arquivo:
        # Repetimos a operação enquanto a condição indicar que a etapa ainda não terminou.
        while bloco := arquivo.read(tamanho_bloco):
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            linhas += bloco.count(b"\n")
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return max(linhas - 1, 0)

# Guardamos em `inventario` a evidência produzida por esta operação.
inventario = []
# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for nome, caminho in sorted(ARQUIVOS.items()):
    # Guardamos em `codificacao` a evidência produzida por esta operação.
    codificacao = "latin1" if nome == "HomeCredit_columns_description.csv" else "utf-8"
    # Guardamos em `cabecalho` a evidência produzida por esta operação.
    cabecalho = pd.read_csv(caminho, nrows=0, encoding=codificacao)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    inventario.append({
        # Registramos `dataset` para manter esta informação nomeada e auditável.
        "dataset": nome,
        # Registramos `linhas` para manter esta informação nomeada e auditável.
        "linhas": contar_linhas_csv(caminho),
        # Registramos `colunas` para manter esta informação nomeada e auditável.
        "colunas": len(cabecalho.columns),
        # Registramos `tamanho_mb` para manter esta informação nomeada e auditável.
        "tamanho_mb": caminho.stat().st_size / 1024**2,
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    })

# Guardamos em `inventario` a evidência produzida por esta operação.
inventario = pd.DataFrame(inventario).sort_values("linhas", ascending=False)
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    inventario.style
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .format({"linhas": "{:,.0f}", "colunas": "{:,.0f}", "tamanho_mb": "{:,.1f}"})
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .background_gradient(subset=["linhas"], cmap="GnBu")
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .hide(axis="index")
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)


### Como as tabelas conversam

<div style="display:grid;grid-template-columns:1fr 1.3fr 1fr;gap:14px;align-items:stretch">
  <div style="background:#F2F7F8;padding:16px;border-radius:12px"><b>Bureau externo</b><br><code>bureau</code><br><code>bureau_balance</code><br><small>Chaves: SK_ID_CURR → SK_ID_BUREAU</small></div>
  <div style="background:#0E4D64;color:white;padding:18px;border-radius:12px;text-align:center"><b>Decisão atual</b><br><code style="color:#B9E3E8">application_train / test</code><br><small>Uma linha por SK_ID_CURR</small></div>
  <div style="background:#F2F7F8;padding:16px;border-radius:12px"><b>Histórico interno</b><br><code>previous_application</code><br><code>POS_CASH</code> • <code>credit_card</code> • <code>installments</code><br><small>Chaves: SK_ID_CURR / SK_ID_PREV</small></div>
</div>

O `TARGET` vale 1 quando houve dificuldade de pagamento. Ele existe somente em `application_train` e nunca será usado na criação de atributos históricos.


In [ ]:
# Guardamos em `app_train` a evidência produzida por esta operação.
app_train = pd.read_csv(ARQUIVOS["application_train.csv"], low_memory=False)
# Guardamos em `app_test` a evidência produzida por esta operação.
app_test = pd.read_csv(ARQUIVOS["application_test.csv"], low_memory=False)

# Guardamos em `dicionario` a evidência produzida por esta operação.
dicionario = None
# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if "HomeCredit_columns_description.csv" in ARQUIVOS:
    # Guardamos em `dicionario` a evidência produzida por esta operação.
    dicionario = pd.read_csv(
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ARQUIVOS["HomeCredit_columns_description.csv"],
        # Guardamos em `encoding` a evidência produzida por esta operação.
        encoding="latin1",
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    )

# Abrimos o portão de qualidade que decidirá se esta etapa está madura para avançar.
portao_qualidade("Etapa 1 — Integridade e relações", {
    # Registramos `application_train não foi carregado` para manter esta informação nomeada e auditável.
    "application_train não foi carregado": len(app_train) > 0,
    # Registramos `application_test não foi carregado` para manter esta informação nomeada e auditável.
    "application_test não foi carregado": len(app_test) > 0,
    # Registramos `SK_ID_CURR não é único no treino` para manter esta informação nomeada e auditável.
    "SK_ID_CURR não é único no treino": app_train["SK_ID_CURR"].is_unique,
    # Registramos `SK_ID_CURR não é único no teste` para manter esta informação nomeada e auditável.
    "SK_ID_CURR não é único no teste": app_test["SK_ID_CURR"].is_unique,
    # Registramos `TARGET contém valores além de 0 e 1` para manter esta informação nomeada e auditável.
    "TARGET contém valores além de 0 e 1": set(app_train["TARGET"].dropna().unique()) == {0, 1},
    # Registramos `Há clientes simultaneamente em treino e teste` para manter esta informação nomeada e auditável.
    "Há clientes simultaneamente em treino e teste": len(set(app_train["SK_ID_CURR"]) & set(app_test["SK_ID_CURR"])) == 0,
    # Registramos `A base completa exigida ainda tem arquivos ausentes` para manter esta informação nomeada e auditável.
    "A base completa exigida ainda tem arquivos ausentes": (not EXIGIR_BASE_COMPLETA) or not faltantes,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})


## 2. O retrato do risco

Antes de prever, precisamos dimensionar o evento. Inadimplência é rara o suficiente para tornar a acurácia enganosa: um modelo que dissesse “ninguém terá dificuldade” acertaria a maioria e seria inútil. Por isso, as métricas centrais serão **ROC AUC**, **PR AUC**, **KS**, sensibilidade e calibração.


In [ ]:
# Guardamos em `taxa_inadimplencia` a evidência produzida por esta operação.
taxa_inadimplencia = app_train["TARGET"].mean()
# Guardamos em `contagem_target` a evidência produzida por esta operação.
contagem_target = app_train["TARGET"].value_counts().sort_index()

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, ax = plt.subplots(figsize=(8.5, 4.5))
# Guardamos em `barras` a evidência produzida por esta operação.
barras = ax.bar(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ["Sem dificuldade", "Com dificuldade"],
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    contagem_target.values,
    # Guardamos em `color` a evidência produzida por esta operação.
    color=[CORES["turquesa"], CORES["coral"]],
    # Guardamos em `width` a evidência produzida por esta operação.
    width=0.58,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set_title("O evento de risco é minoritário — e isso muda a avaliação")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set_ylabel("Clientes")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.grid(axis="x", visible=False)
# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for barra, valor in zip(barras, contagem_target.values):
    # Refinamos o gráfico para que a mensagem visual seja direta e elegante.
    ax.text(barra.get_x() + barra.get_width()/2, valor, br_numero(valor), ha="center", va="bottom", fontweight="bold")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.text(0.99, 0.93, f"Taxa de inadimplência: {taxa_inadimplencia:.2%}", transform=ax.transAxes, ha="right", color=CORES["coral"], fontweight="bold")
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(Markdown(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f"**Leitura executiva.** Há **{br_numero(contagem_target.get(1, 0))}** clientes com dificuldade "
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f"entre **{br_numero(len(app_train))}** observações — uma prevalência de **{taxa_inadimplencia:.2%}**."
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))


In [ ]:
# Guardamos em `ausencias` a evidência produzida por esta operação.
ausencias = app_train.isna().mean().sort_values(ascending=False)
# Guardamos em `top_ausencias` a evidência produzida por esta operação.
top_ausencias = ausencias.head(20).sort_values()
# Guardamos em `anomalia_emprego` a evidência produzida por esta operação.
anomalia_emprego = int((app_train["DAYS_EMPLOYED"] == 365243).sum())

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, ax = plt.subplots(figsize=(9, 6.2))
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.barh(top_ausencias.index, top_ausencias.values * 100, color=CORES["azul"])
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set_title("Concentração de ausências: características do imóvel dominam o topo")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set_xlabel("Valores ausentes (%)")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set_ylabel("")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0f}%")
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()

# Guardamos em `resumo_qualidade` a evidência produzida por esta operação.
resumo_qualidade = pd.DataFrame({
    # Registramos `indicador` para manter esta informação nomeada e auditável.
    "indicador": ["Células ausentes", "Colunas com > 65% de ausência", "Sentinela 365243 em DAYS_EMPLOYED"],
    # Registramos `valor` para manter esta informação nomeada e auditável.
    "valor": [
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        f"{app_train.isna().sum().sum() / app_train.size:.2%}",
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        int((ausencias > 0.65).sum()),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        br_numero(anomalia_emprego),
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ],
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(resumo_qualidade.style.hide(axis="index"))


In [ ]:
# Guardamos em `eda` a evidência produzida por esta operação.
eda = app_train[[
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "TARGET", "DAYS_BIRTH", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "NAME_CONTRACT_TYPE",
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
]].copy()
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
eda["IDADE_ANOS"] = -eda["DAYS_BIRTH"] / 365.25
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
eda["MEDIA_FONTES_EXTERNAS"] = eda[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
eda["FAIXA_ETARIA"] = pd.cut(eda["IDADE_ANOS"], [20, 30, 40, 50, 60, 70], right=False)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
eda["FAIXA_RENDA"] = pd.qcut(eda["AMT_INCOME_TOTAL"].rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Guardamos em `taxa_idade` a evidência produzida por esta operação.
taxa_idade = eda.groupby("FAIXA_ETARIA", observed=True)["TARGET"].mean()
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
taxa_idade.plot(kind="bar", ax=axes[0,0], color=CORES["azul"], rot=0)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0,0].set_title("Risco observado por faixa etária")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0,0].set_xlabel("Idade")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0,0].set_ylabel("Taxa de dificuldade")

# Guardamos em `taxa_renda` a evidência produzida por esta operação.
taxa_renda = eda.groupby("FAIXA_RENDA", observed=True)["TARGET"].mean()
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
taxa_renda.plot(kind="bar", ax=axes[0,1], color=CORES["turquesa"], rot=0)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0,1].set_title("Risco observado por quintil de renda")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0,1].set_xlabel("Quintil de renda")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0,1].set_ylabel("")

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.kdeplot(data=eda.sample(min(100_000, len(eda)), random_state=RANDOM_STATE), x="MEDIA_FONTES_EXTERNAS", hue="TARGET", common_norm=False, fill=False, palette=[CORES["turquesa"], CORES["coral"]], ax=axes[1,0])
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1,0].set_title("Fontes externas separam parte do risco")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1,0].set_xlabel("Média das fontes externas")

# Guardamos em `taxa_contrato` a evidência produzida por esta operação.
taxa_contrato = eda.groupby("NAME_CONTRACT_TYPE")["TARGET"].mean().sort_values()
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
taxa_contrato.plot(kind="barh", ax=axes[1,1], color=CORES["dourado"])
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1,1].set_title("Risco por modalidade do contrato")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1,1].set_xlabel("Taxa de dificuldade")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1,1].set_ylabel("")

# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for ax in axes.flat:
    # Refinamos o gráfico para que a mensagem visual seja direta e elegante.
    ax.yaxis.set_major_formatter(lambda y, pos: f"{y:.0%}" if y < 1 else f"{y:.0f}")
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.tight_layout()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()

# Guardamos em `media_ext` a evidência produzida por esta operação.
media_ext = eda.groupby("TARGET")["MEDIA_FONTES_EXTERNAS"].mean()
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(Markdown(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f"**Primeiro sinal forte.** A média das fontes externas cai de **{media_ext.get(0, np.nan):.3f}** "
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f"entre clientes sem dificuldade para **{media_ext.get(1, np.nan):.3f}** entre os inadimplentes. "
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "A relação é descritiva, não causal."
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))


In [ ]:
# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if dicionario is not None:
    # Guardamos em `tabela_coluna` a evidência produzida por esta operação.
    tabela_coluna = next((c for c in dicionario.columns if c.lower() == "table"), None)
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if tabela_coluna:
        # Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
        display(
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            dicionario.groupby(tabela_coluna, dropna=False)
            # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
            .size().rename("variaveis_documentadas")
            # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
            .sort_values(ascending=False).to_frame()
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        )
    # Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
    display(dicionario.head(8))

# Abrimos o portão de qualidade que decidirá se esta etapa está madura para avançar.
portao_qualidade("Etapa 2 — Entendimento e qualidade", {
    # Registramos `A taxa do evento não foi calculada` para manter esta informação nomeada e auditável.
    "A taxa do evento não foi calculada": 0 < taxa_inadimplencia < 1,
    # Registramos `As ausências não foram quantificadas` para manter esta informação nomeada e auditável.
    "As ausências não foram quantificadas": len(ausencias) == app_train.shape[1],
    # Registramos `A sentinela de DAYS_EMPLOYED não foi identificada` para manter esta informação nomeada e auditável.
    "A sentinela de DAYS_EMPLOYED não foi identificada": anomalia_emprego > 0,
    # Registramos `As fontes externas não foram comparadas por TARGET` para manter esta informação nomeada e auditável.
    "As fontes externas não foram comparadas por TARGET": media_ext.notna().all(),
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})


## 3. A memória financeira do cliente

Uma linha da proposta atual não conta toda a história. Vamos converter milhões de registros históricos em atributos simples por cliente: quantidade de contratos, exposição, saldo, utilização de limite, atraso, severidade e recência. Não usamos o `TARGET` em nenhuma agregação.

| Dataset | Grão original | Sinais extraídos |
|---|---|---|
| `bureau` | um crédito em outra instituição | exposição, dívida, vencidos, atividade e recência |
| `bureau_balance` | um mês de um crédito do bureau | meses observados e severidade de atraso |
| `previous_application` | uma proposta anterior interna | aprovações, recusas, valores e recência |
| `POS_CASH_balance` | um mês de contrato POS/CASH | atraso, parcelas futuras e conclusão |
| `credit_card_balance` | um mês de cartão | utilização, saldo, pagamentos e atraso |
| `installments_payments` | uma parcela/pagamento | atraso de pagamento e pagamento insuficiente |


In [ ]:
# Criamos a função `caminho` para transformar esta ideia em uma etapa reutilizável.
def caminho(nome):
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return ARQUIVOS.get(nome)

# Criamos a função `divisao_segura` para transformar esta ideia em uma etapa reutilizável.
def divisao_segura(numerador, denominador):
    # Guardamos em `resultado` a evidência produzida por esta operação.
    resultado = numerador / denominador.replace(0, np.nan)
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return resultado.replace([np.inf, -np.inf], np.nan)

# Criamos a função `agregar_bureau` para transformar esta ideia em uma etapa reutilizável.
def agregar_bureau():
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if caminho("bureau.csv") is None:
        # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
        return None
    # Guardamos em `bureau` a evidência produzida por esta operação.
    bureau = pd.read_csv(caminho("bureau.csv"), usecols=[
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "AMT_CREDIT_SUM_OVERDUE",
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "DAYS_CREDIT", "CNT_CREDIT_PROLONG",
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ], low_memory=False)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    bureau["BUREAU_ATIVO"] = (bureau["CREDIT_ACTIVE"] == "Active").astype("int8")
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    bureau["BUREAU_COM_ATRASO"] = (bureau["AMT_CREDIT_SUM_OVERDUE"].fillna(0) > 0).astype("int8")

    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if caminho("bureau_balance.csv") is not None:
        # Guardamos em `saldo` a evidência produzida por esta operação.
        saldo = pd.read_csv(
            # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
            caminho("bureau_balance.csv"),
            # Guardamos em `dtype` a evidência produzida por esta operação.
            dtype={"SK_ID_BUREAU": "int32", "MONTHS_BALANCE": "int16", "STATUS": "category"},
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        )
        # Guardamos em `mapa_status` a evidência produzida por esta operação.
        mapa_status = {"X": 0, "C": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        saldo["BB_STATUS_NUM"] = saldo["STATUS"].map(mapa_status).astype("float32")
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        saldo["BB_MES_EM_ATRASO"] = saldo["STATUS"].isin(["1", "2", "3", "4", "5"]).astype("int8")
        # Guardamos em `saldo_cliente` a evidência produzida por esta operação.
        saldo_cliente = saldo.groupby("SK_ID_BUREAU", observed=True).agg(
            # Fixamos `BB_MESES` como uma decisão explícita e fácil de ajustar pelo leitor.
            BB_MESES=("MONTHS_BALANCE", "count"),
            # Fixamos `BB_MES_MAIS_RECENTE` como uma decisão explícita e fácil de ajustar pelo leitor.
            BB_MES_MAIS_RECENTE=("MONTHS_BALANCE", "max"),
            # Fixamos `BB_MESES_EM_ATRASO` como uma decisão explícita e fácil de ajustar pelo leitor.
            BB_MESES_EM_ATRASO=("BB_MES_EM_ATRASO", "sum"),
            # Fixamos `BB_PIOR_STATUS` como uma decisão explícita e fácil de ajustar pelo leitor.
            BB_PIOR_STATUS=("BB_STATUS_NUM", "max"),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ).reset_index()
        # Guardamos em `bureau` a evidência produzida por esta operação.
        bureau = bureau.merge(saldo_cliente, on="SK_ID_BUREAU", how="left", validate="one_to_one")
        # Liberamos da memória o que já cumpriu seu papel para manter o Colab leve.
        del saldo, saldo_cliente

    # Guardamos em `especificacao` a evidência produzida por esta operação.
    especificacao = {
        # Registramos `BUREAU_CONTRATOS` para manter esta informação nomeada e auditável.
        "BUREAU_CONTRATOS": ("SK_ID_BUREAU", "nunique"),
        # Registramos `BUREAU_ATIVOS` para manter esta informação nomeada e auditável.
        "BUREAU_ATIVOS": ("BUREAU_ATIVO", "sum"),
        # Registramos `BUREAU_CONTRATOS_COM_ATRASO` para manter esta informação nomeada e auditável.
        "BUREAU_CONTRATOS_COM_ATRASO": ("BUREAU_COM_ATRASO", "sum"),
        # Registramos `BUREAU_CREDITO_TOTAL` para manter esta informação nomeada e auditável.
        "BUREAU_CREDITO_TOTAL": ("AMT_CREDIT_SUM", "sum"),
        # Registramos `BUREAU_DIVIDA_TOTAL` para manter esta informação nomeada e auditável.
        "BUREAU_DIVIDA_TOTAL": ("AMT_CREDIT_SUM_DEBT", "sum"),
        # Registramos `BUREAU_VENCIDO_TOTAL` para manter esta informação nomeada e auditável.
        "BUREAU_VENCIDO_TOTAL": ("AMT_CREDIT_SUM_OVERDUE", "sum"),
        # Registramos `BUREAU_DIAS_CREDITO_MEDIO` para manter esta informação nomeada e auditável.
        "BUREAU_DIAS_CREDITO_MEDIO": ("DAYS_CREDIT", "mean"),
        # Registramos `BUREAU_PROLONGAMENTOS` para manter esta informação nomeada e auditável.
        "BUREAU_PROLONGAMENTOS": ("CNT_CREDIT_PROLONG", "sum"),
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    }
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if "BB_MESES" in bureau:
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        especificacao.update({
            # Registramos `BB_MESES_OBSERVADOS` para manter esta informação nomeada e auditável.
            "BB_MESES_OBSERVADOS": ("BB_MESES", "sum"),
            # Registramos `BB_MESES_EM_ATRASO` para manter esta informação nomeada e auditável.
            "BB_MESES_EM_ATRASO": ("BB_MESES_EM_ATRASO", "sum"),
            # Registramos `BB_PIOR_STATUS` para manter esta informação nomeada e auditável.
            "BB_PIOR_STATUS": ("BB_PIOR_STATUS", "max"),
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        })
    # Guardamos em `agregado` a evidência produzida por esta operação.
    agregado = bureau.groupby("SK_ID_CURR").agg(**especificacao).reset_index()
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    agregado["BUREAU_RAZAO_DIVIDA_CREDITO"] = divisao_segura(
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        agregado["BUREAU_DIVIDA_TOTAL"], agregado["BUREAU_CREDITO_TOTAL"]
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    )
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return agregado

# Criamos a função `agregar_previous_application` para transformar esta ideia em uma etapa reutilizável.
def agregar_previous_application():
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if caminho("previous_application.csv") is None:
        # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
        return None
    # Guardamos em `anterior` a evidência produzida por esta operação.
    anterior = pd.read_csv(caminho("previous_application.csv"), usecols=[
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "SK_ID_CURR", "SK_ID_PREV", "NAME_CONTRACT_STATUS", "AMT_APPLICATION",
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "AMT_CREDIT", "AMT_ANNUITY", "AMT_DOWN_PAYMENT", "CNT_PAYMENT", "DAYS_DECISION",
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ], low_memory=False)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    anterior["PREV_APROVADA"] = (anterior["NAME_CONTRACT_STATUS"] == "Approved").astype("int8")
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    anterior["PREV_RECUSADA"] = (anterior["NAME_CONTRACT_STATUS"] == "Refused").astype("int8")
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    anterior["PREV_RAZAO_CREDITO_PEDIDO"] = divisao_segura(anterior["AMT_CREDIT"], anterior["AMT_APPLICATION"])
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return anterior.groupby("SK_ID_CURR").agg(
        # Fixamos `PREV_PROPOSTAS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_PROPOSTAS=("SK_ID_PREV", "nunique"),
        # Fixamos `PREV_APROVADAS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_APROVADAS=("PREV_APROVADA", "sum"),
        # Fixamos `PREV_RECUSADAS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_RECUSADAS=("PREV_RECUSADA", "sum"),
        # Fixamos `PREV_CREDITO_MEDIO` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_CREDITO_MEDIO=("AMT_CREDIT", "mean"),
        # Fixamos `PREV_ANUIDADE_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_ANUIDADE_MEDIA=("AMT_ANNUITY", "mean"),
        # Fixamos `PREV_ENTRADA_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_ENTRADA_MEDIA=("AMT_DOWN_PAYMENT", "mean"),
        # Fixamos `PREV_PARCELAS_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_PARCELAS_MEDIA=("CNT_PAYMENT", "mean"),
        # Fixamos `PREV_RAZAO_CREDITO_PEDIDO_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_RAZAO_CREDITO_PEDIDO_MEDIA=("PREV_RAZAO_CREDITO_PEDIDO", "mean"),
        # Fixamos `PREV_DECISAO_MAIS_RECENTE` como uma decisão explícita e fácil de ajustar pelo leitor.
        PREV_DECISAO_MAIS_RECENTE=("DAYS_DECISION", "max"),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ).reset_index()

# Criamos a função `agregar_pos_cash` para transformar esta ideia em uma etapa reutilizável.
def agregar_pos_cash():
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if caminho("POS_CASH_balance.csv") is None:
        # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
        return None
    # Guardamos em `pos` a evidência produzida por esta operação.
    pos = pd.read_csv(caminho("POS_CASH_balance.csv"), usecols=[
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE", "CNT_INSTALMENT_FUTURE",
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "SK_DPD", "SK_DPD_DEF", "NAME_CONTRACT_STATUS",
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ], low_memory=False)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    pos["POS_EM_ATRASO"] = (pos["SK_DPD"].fillna(0) > 0).astype("int8")
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    pos["POS_CONCLUIDO"] = (pos["NAME_CONTRACT_STATUS"] == "Completed").astype("int8")
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return pos.groupby("SK_ID_CURR").agg(
        # Fixamos `POS_CONTRATOS` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_CONTRATOS=("SK_ID_PREV", "nunique"),
        # Fixamos `POS_MESES` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_MESES=("MONTHS_BALANCE", "count"),
        # Fixamos `POS_MES_MAIS_RECENTE` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_MES_MAIS_RECENTE=("MONTHS_BALANCE", "max"),
        # Fixamos `POS_PARCELAS_FUTURAS_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_PARCELAS_FUTURAS_MEDIA=("CNT_INSTALMENT_FUTURE", "mean"),
        # Fixamos `POS_DPD_MEDIO` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_DPD_MEDIO=("SK_DPD", "mean"),
        # Fixamos `POS_DPD_MAXIMO` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_DPD_MAXIMO=("SK_DPD", "max"),
        # Fixamos `POS_DPD_DEF_MEDIO` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_DPD_DEF_MEDIO=("SK_DPD_DEF", "mean"),
        # Fixamos `POS_MESES_EM_ATRASO` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_MESES_EM_ATRASO=("POS_EM_ATRASO", "sum"),
        # Fixamos `POS_MESES_CONCLUIDOS` como uma decisão explícita e fácil de ajustar pelo leitor.
        POS_MESES_CONCLUIDOS=("POS_CONCLUIDO", "sum"),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ).reset_index()

# Criamos a função `agregar_cartao` para transformar esta ideia em uma etapa reutilizável.
def agregar_cartao():
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if caminho("credit_card_balance.csv") is None:
        # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
        return None
    # Guardamos em `cartao` a evidência produzida por esta operação.
    cartao = pd.read_csv(caminho("credit_card_balance.csv"), usecols=[
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE", "AMT_BALANCE",
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT", "SK_DPD",
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ], low_memory=False)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    cartao["CC_UTILIZACAO"] = divisao_segura(cartao["AMT_BALANCE"], cartao["AMT_CREDIT_LIMIT_ACTUAL"])
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    cartao["CC_EM_ATRASO"] = (cartao["SK_DPD"].fillna(0) > 0).astype("int8")
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return cartao.groupby("SK_ID_CURR").agg(
        # Fixamos `CC_CONTRATOS` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_CONTRATOS=("SK_ID_PREV", "nunique"),
        # Fixamos `CC_MESES` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_MESES=("MONTHS_BALANCE", "count"),
        # Fixamos `CC_SALDO_MEDIO` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_SALDO_MEDIO=("AMT_BALANCE", "mean"),
        # Fixamos `CC_SALDO_MAXIMO` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_SALDO_MAXIMO=("AMT_BALANCE", "max"),
        # Fixamos `CC_LIMITE_MEDIO` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_LIMITE_MEDIO=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
        # Fixamos `CC_UTILIZACAO_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_UTILIZACAO_MEDIA=("CC_UTILIZACAO", "mean"),
        # Fixamos `CC_UTILIZACAO_MAXIMA` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_UTILIZACAO_MAXIMA=("CC_UTILIZACAO", "max"),
        # Fixamos `CC_SAQUES_MEDIOS` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_SAQUES_MEDIOS=("AMT_DRAWINGS_CURRENT", "mean"),
        # Fixamos `CC_PAGAMENTO_MEDIO` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_PAGAMENTO_MEDIO=("AMT_PAYMENT_CURRENT", "mean"),
        # Fixamos `CC_DPD_MAXIMO` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_DPD_MAXIMO=("SK_DPD", "max"),
        # Fixamos `CC_MESES_EM_ATRASO` como uma decisão explícita e fácil de ajustar pelo leitor.
        CC_MESES_EM_ATRASO=("CC_EM_ATRASO", "sum"),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ).reset_index()

# Criamos a função `agregar_parcelas` para transformar esta ideia em uma etapa reutilizável.
def agregar_parcelas():
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if caminho("installments_payments.csv") is None:
        # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
        return None
    # Guardamos em `parcelas` a evidência produzida por esta operação.
    parcelas = pd.read_csv(caminho("installments_payments.csv"), usecols=[
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT",
        # Incluímos este elemento na coleção que organiza os componentes da etapa.
        "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ], low_memory=False)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    parcelas["PARC_ATRASO_DIAS"] = (parcelas["DAYS_ENTRY_PAYMENT"] - parcelas["DAYS_INSTALMENT"]).clip(lower=0)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    parcelas["PARC_EM_ATRASO"] = (parcelas["PARC_ATRASO_DIAS"] > 0).astype("int8")
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    parcelas["PARC_RAZAO_PAGA"] = divisao_segura(parcelas["AMT_PAYMENT"], parcelas["AMT_INSTALMENT"])
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    parcelas["PARC_PAGAMENTO_INSUFICIENTE"] = (parcelas["PARC_RAZAO_PAGA"] < 0.99).astype("int8")
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return parcelas.groupby("SK_ID_CURR").agg(
        # Fixamos `PARC_CONTRATOS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_CONTRATOS=("SK_ID_PREV", "nunique"),
        # Fixamos `PARC_REGISTROS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_REGISTROS=("NUM_INSTALMENT_NUMBER", "count"),
        # Fixamos `PARC_ATRASO_MEDIO_DIAS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_ATRASO_MEDIO_DIAS=("PARC_ATRASO_DIAS", "mean"),
        # Fixamos `PARC_ATRASO_MAXIMO_DIAS` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_ATRASO_MAXIMO_DIAS=("PARC_ATRASO_DIAS", "max"),
        # Fixamos `PARC_PARCELAS_EM_ATRASO` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_PARCELAS_EM_ATRASO=("PARC_EM_ATRASO", "sum"),
        # Fixamos `PARC_RAZAO_PAGA_MEDIA` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_RAZAO_PAGA_MEDIA=("PARC_RAZAO_PAGA", "mean"),
        # Fixamos `PARC_PAGAMENTOS_INSUFICIENTES` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_PAGAMENTOS_INSUFICIENTES=("PARC_PAGAMENTO_INSUFICIENTE", "sum"),
        # Fixamos `PARC_VALOR_PREVISTO` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_VALOR_PREVISTO=("AMT_INSTALMENT", "sum"),
        # Fixamos `PARC_VALOR_PAGO` como uma decisão explícita e fácil de ajustar pelo leitor.
        PARC_VALOR_PAGO=("AMT_PAYMENT", "sum"),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ).reset_index()


In [ ]:
# Criamos a função `criar_features_aplicacao` para transformar esta ideia em uma etapa reutilizável.
def criar_features_aplicacao(dados):
    # Guardamos em `df` a evidência produzida por esta operação.
    df = dados.copy()
    # Guardamos em `emprego` a evidência produzida por esta operação.
    emprego = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["IDADE_ANOS"] = -df["DAYS_BIRTH"] / 365.25
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["TEMPO_EMPREGO_ANOS"] = -emprego / 365.25
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["TEMPO_RESIDENCIA_ANOS"] = -df["DAYS_REGISTRATION"] / 365.25
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["RAZAO_CREDITO_RENDA"] = divisao_segura(df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"])
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["RAZAO_ANUIDADE_RENDA"] = divisao_segura(df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"])
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["RAZAO_CREDITO_BEM"] = divisao_segura(df["AMT_CREDIT"], df["AMT_GOODS_PRICE"])
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["RENDA_POR_PESSOA"] = divisao_segura(df["AMT_INCOME_TOTAL"], df["CNT_FAM_MEMBERS"])
    # Guardamos em `fontes` a evidência produzida por esta operação.
    fontes = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df]
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["MEDIA_FONTES_EXTERNAS"] = df[fontes].mean(axis=1)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    df["QTD_FONTES_EXTERNAS"] = df[fontes].notna().sum(axis=1)
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return df

# Guardamos em `treino_modelo` a evidência produzida por esta operação.
treino_modelo = criar_features_aplicacao(app_train)
# Guardamos em `teste_kaggle` a evidência produzida por esta operação.
teste_kaggle = criar_features_aplicacao(app_test)
# Liberamos da memória o que já cumpriu seu papel para manter o Colab leve.
del app_train, app_test, eda

# Guardamos em `agregadores` a evidência produzida por esta operação.
agregadores = {
    # Registramos `bureau + bureau_balance` para manter esta informação nomeada e auditável.
    "bureau + bureau_balance": agregar_bureau,
    # Registramos `previous_application` para manter esta informação nomeada e auditável.
    "previous_application": agregar_previous_application,
    # Registramos `POS_CASH_balance` para manter esta informação nomeada e auditável.
    "POS_CASH_balance": agregar_pos_cash,
    # Registramos `credit_card_balance` para manter esta informação nomeada e auditável.
    "credit_card_balance": agregar_cartao,
    # Registramos `installments_payments` para manter esta informação nomeada e auditável.
    "installments_payments": agregar_parcelas,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
}

# Guardamos em `tabelas_features` a evidência produzida por esta operação.
tabelas_features = []
# Guardamos em `resumo_agregacoes` a evidência produzida por esta operação.
resumo_agregacoes = []
# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for nome, funcao in agregadores.items():
    # Guardamos em `inicio` a evidência produzida por esta operação.
    inicio = time.time()
    # Guardamos em `tabela` a evidência produzida por esta operação.
    tabela = funcao()
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if tabela is None:
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        resumo_agregacoes.append({"fonte": nome, "status": "ausente", "clientes": 0, "atributos": 0, "segundos": 0})
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        continue
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if not tabela["SK_ID_CURR"].is_unique:
        # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
        raise ValueError(f"A agregação {nome} não ficou única por cliente.")
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    tabelas_features.append(tabela)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    resumo_agregacoes.append({
        # Registramos `fonte` para manter esta informação nomeada e auditável.
        "fonte": nome,
        # Registramos `status` para manter esta informação nomeada e auditável.
        "status": "processada",
        # Registramos `clientes` para manter esta informação nomeada e auditável.
        "clientes": len(tabela),
        # Registramos `atributos` para manter esta informação nomeada e auditável.
        "atributos": tabela.shape[1] - 1,
        # Registramos `segundos` para manter esta informação nomeada e auditável.
        "segundos": time.time() - inicio,
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    })

# Guardamos em `resumo_agregacoes` a evidência produzida por esta operação.
resumo_agregacoes = pd.DataFrame(resumo_agregacoes)
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(resumo_agregacoes.style.format({"clientes": "{:,.0f}", "segundos": "{:,.1f}"}).hide(axis="index"))


In [ ]:
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
linhas_treino_antes, linhas_teste_antes = len(treino_modelo), len(teste_kaggle)
# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for tabela in tabelas_features:
    # Guardamos em `treino_modelo` a evidência produzida por esta operação.
    treino_modelo = treino_modelo.merge(tabela, on="SK_ID_CURR", how="left", validate="one_to_one")
    # Guardamos em `teste_kaggle` a evidência produzida por esta operação.
    teste_kaggle = teste_kaggle.merge(tabela, on="SK_ID_CURR", how="left", validate="one_to_one")

# Guardamos em `atributos_historicos` a evidência produzida por esta operação.
atributos_historicos = sum(t.shape[1] - 1 for t in tabelas_features)
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(Markdown(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f"Foram criados **{atributos_historicos} atributos históricos** e a tabela analítica final possui "
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    f"**{treino_modelo.shape[1] - 1} variáveis candidatas** para **{br_numero(len(treino_modelo))} clientes**."
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))

# Abrimos o portão de qualidade que decidirá se esta etapa está madura para avançar.
portao_qualidade("Etapa 3 — Engenharia de atributos", {
    # Registramos `Alguma agregação não ficou única por cliente` para manter esta informação nomeada e auditável.
    "Alguma agregação não ficou única por cliente": all(t["SK_ID_CURR"].is_unique for t in tabelas_features),
    # Registramos `O merge alterou a quantidade de clientes no treino` para manter esta informação nomeada e auditável.
    "O merge alterou a quantidade de clientes no treino": len(treino_modelo) == linhas_treino_antes,
    # Registramos `O merge alterou a quantidade de clientes no teste` para manter esta informação nomeada e auditável.
    "O merge alterou a quantidade de clientes no teste": len(teste_kaggle) == linhas_teste_antes,
    # Registramos `Nenhum atributo histórico foi criado` para manter esta informação nomeada e auditável.
    "Nenhum atributo histórico foi criado": atributos_historicos > 0,
    # Registramos `O TARGET foi introduzido em tabela histórica` para manter esta informação nomeada e auditável.
    "O TARGET foi introduzido em tabela histórica": all("TARGET" not in t.columns for t in tabelas_features),
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})


## 4. Do dado ao modelo — sem vazamento

O `application_test` do Kaggle não tem rótulo e serve apenas para a submissão final. Para avaliar honestamente, separamos o `application_train` em três partes estratificadas:

- **Treino (70%)**: ajusta os parâmetros dos modelos;
- **Validação (15%)**: escolhe o modelo e o limiar operacional;
- **Teste interno (15%)**: mede o desempenho final uma única vez.

`CODE_GENDER` é preservado somente para auditoria de equidade e não entra no modelo. Colunas com mais de 65% de ausência, identificadores e constantes também são removidas.


In [ ]:
# Fixamos `ALVO` como uma decisão explícita e fácil de ajustar pelo leitor.
ALVO = "TARGET"
# Fixamos `IDENTIFICADORES` como uma decisão explícita e fácil de ajustar pelo leitor.
IDENTIFICADORES = ["SK_ID_CURR"]
# Fixamos `SENSIVEIS_AUDITORIA` como uma decisão explícita e fácil de ajustar pelo leitor.
SENSIVEIS_AUDITORIA = ["CODE_GENDER"]

# Guardamos em `y_total` a evidência produzida por esta operação.
y_total = treino_modelo[ALVO].astype("int8")
# Guardamos em `X_total` a evidência produzida por esta operação.
X_total = treino_modelo.drop(columns=[ALVO])
# Guardamos em `X_kaggle` a evidência produzida por esta operação.
X_kaggle = teste_kaggle.copy()

# Guardamos em `limite_ausencia` a evidência produzida por esta operação.
limite_ausencia = 0.65
# Guardamos em `colunas_muito_ausentes` a evidência produzida por esta operação.
colunas_muito_ausentes = X_total.columns[X_total.isna().mean() > limite_ausencia].tolist()
# Guardamos em `colunas_constantes` a evidência produzida por esta operação.
colunas_constantes = [c for c in X_total.columns if X_total[c].nunique(dropna=False) <= 1]
# Guardamos em `colunas_excluir` a evidência produzida por esta operação.
colunas_excluir = sorted(set(IDENTIFICADORES + SENSIVEIS_AUDITORIA + colunas_muito_ausentes + colunas_constantes))

# Guardamos em `X_total` a evidência produzida por esta operação.
X_total = X_total.drop(columns=colunas_excluir, errors="ignore").replace([np.inf, -np.inf], np.nan)
# Guardamos em `X_kaggle` a evidência produzida por esta operação.
X_kaggle = X_kaggle.reindex(columns=X_total.columns).replace([np.inf, -np.inf], np.nan)

# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if MODO_EXECUCAO == "rapido" and len(X_total) > 80_000:
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    X_total, _, y_total, _ = train_test_split(
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        X_total, y_total, train_size=80_000, stratify=y_total, random_state=RANDOM_STATE
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    )
    # Contamos ao leitor o estado atual da execução para que nenhum passo aconteça no escuro.
    print("Modo rápido: modelagem limitada a 80.000 clientes, com estratificação.")

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
X_desenvolvimento, X_teste, y_desenvolvimento, y_teste = train_test_split(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    X_total, y_total, test_size=0.15, stratify=y_total, random_state=RANDOM_STATE
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    X_desenvolvimento,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    y_desenvolvimento,
    # Guardamos em `test_size` a evidência produzida por esta operação.
    test_size=0.1764705882,
    # Guardamos em `stratify` a evidência produzida por esta operação.
    stratify=y_desenvolvimento,
    # Guardamos em `random_state` a evidência produzida por esta operação.
    random_state=RANDOM_STATE,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)

# Guardamos em `colunas_numericas` a evidência produzida por esta operação.
colunas_numericas = X_treino.select_dtypes(include=np.number).columns.tolist()
# Guardamos em `colunas_categoricas` a evidência produzida por esta operação.
colunas_categoricas = X_treino.select_dtypes(exclude=np.number).columns.tolist()

# Contamos ao leitor o estado atual da execução para que nenhum passo aconteça no escuro.
print(f"Treino: {len(X_treino):,} | Validação: {len(X_validacao):,} | Teste interno: {len(X_teste):,}")
# Contamos ao leitor o estado atual da execução para que nenhum passo aconteça no escuro.
print(f"Variáveis numéricas: {len(colunas_numericas)} | categóricas: {len(colunas_categoricas)}")


In [ ]:
# Guardamos em `preprocessamento_logistico` a evidência produzida por esta operação.
preprocessamento_logistico = ColumnTransformer([
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ("numericas", Pipeline([
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("imputacao", SimpleImputer(strategy="median", add_indicator=True)),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("escala", StandardScaler(with_mean=False)),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ]), colunas_numericas),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ("categoricas", Pipeline([
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("imputacao", SimpleImputer(strategy="most_frequent")),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("one_hot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ]), colunas_categoricas),
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
])

# Guardamos em `preprocessamento_arvore` a evidência produzida por esta operação.
preprocessamento_arvore = ColumnTransformer([
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ("numericas", SimpleImputer(strategy="median", add_indicator=True), colunas_numericas),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ("categoricas", Pipeline([
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("imputacao", SimpleImputer(strategy="most_frequent")),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ]), colunas_categoricas),
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
])

# Guardamos em `modelos` a evidência produzida por esta operação.
modelos = {
    # Registramos `Regressão logística` para manter esta informação nomeada e auditável.
    "Regressão logística": Pipeline([
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("preprocessamento", preprocessamento_logistico),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("modelo", LogisticRegression(C=0.25, max_iter=350, solver="liblinear", random_state=RANDOM_STATE)),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ]),
    # Registramos `Gradient boosting` para manter esta informação nomeada e auditável.
    "Gradient boosting": Pipeline([
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("preprocessamento", preprocessamento_arvore),
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        ("modelo", HistGradientBoostingClassifier(
            # Guardamos em `learning_rate` a evidência produzida por esta operação.
            learning_rate=0.06,
            # Guardamos em `max_iter` a evidência produzida por esta operação.
            max_iter=180,
            # Guardamos em `max_leaf_nodes` a evidência produzida por esta operação.
            max_leaf_nodes=31,
            # Guardamos em `l2_regularization` a evidência produzida por esta operação.
            l2_regularization=1.0,
            # Guardamos em `early_stopping` a evidência produzida por esta operação.
            early_stopping=True,
            # Guardamos em `random_state` a evidência produzida por esta operação.
            random_state=RANDOM_STATE,
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        )),
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    ]),
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
}

# Criamos a função `calcular_metricas` para transformar esta ideia em uma etapa reutilizável.
def calcular_metricas(y_real, probabilidades, limiar=0.5):
    # Guardamos em `classe` a evidência produzida por esta operação.
    classe = (probabilidades >= limiar).astype(int)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    fpr, tpr, _ = roc_curve(y_real, probabilidades)
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return {
        # Registramos `roc_auc` para manter esta informação nomeada e auditável.
        "roc_auc": roc_auc_score(y_real, probabilidades),
        # Registramos `pr_auc` para manter esta informação nomeada e auditável.
        "pr_auc": average_precision_score(y_real, probabilidades),
        # Registramos `ks` para manter esta informação nomeada e auditável.
        "ks": np.max(tpr - fpr),
        # Registramos `brier` para manter esta informação nomeada e auditável.
        "brier": brier_score_loss(y_real, probabilidades),
        # Registramos `sensibilidade` para manter esta informação nomeada e auditável.
        "sensibilidade": recall_score(y_real, classe, zero_division=0),
        # Registramos `precisao` para manter esta informação nomeada e auditável.
        "precisao": precision_score(y_real, classe, zero_division=0),
        # Registramos `f1` para manter esta informação nomeada e auditável.
        "f1": f1_score(y_real, classe, zero_division=0),
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    }


In [ ]:
# Guardamos em `resultados_validacao` a evidência produzida por esta operação.
resultados_validacao = []
# Guardamos em `probabilidades_validacao` a evidência produzida por esta operação.
probabilidades_validacao = {}

# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for nome, modelo in modelos.items():
    # Guardamos em `inicio` a evidência produzida por esta operação.
    inicio = time.time()
    # Ensinamos o modelo usando somente as observações reservadas para aprendizagem.
    modelo.fit(X_treino, y_treino)
    # Pedimos ao modelo probabilidades de risco, preservando a riqueza do score contínuo.
    prob = modelo.predict_proba(X_validacao)[:, 1]
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    probabilidades_validacao[nome] = prob
    # Guardamos em `metricas` a evidência produzida por esta operação.
    metricas = calcular_metricas(y_validacao, prob)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    resultados_validacao.append({"modelo": nome, **metricas, "tempo_segundos": time.time() - inicio})

# Guardamos em `comparacao_modelos` a evidência produzida por esta operação.
comparacao_modelos = pd.DataFrame(resultados_validacao).sort_values("roc_auc", ascending=False)
# Guardamos em `nome_vencedor` a evidência produzida por esta operação.
nome_vencedor = comparacao_modelos.iloc[0]["modelo"]
# Guardamos em `modelo_vencedor` a evidência produzida por esta operação.
modelo_vencedor = modelos[nome_vencedor]
# Guardamos em `prob_validacao` a evidência produzida por esta operação.
prob_validacao = probabilidades_validacao[nome_vencedor]

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    comparacao_modelos.style
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .format({c: "{:.4f}" for c in ["roc_auc", "pr_auc", "ks", "brier", "sensibilidade", "precisao", "f1"]})
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .format({"tempo_segundos": "{:,.1f}"})
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .background_gradient(subset=["roc_auc", "pr_auc", "ks"], cmap="GnBu")
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .hide(axis="index")
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(Markdown(f"**Modelo selecionado na validação:** {nome_vencedor}."))


In [ ]:
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for nome, prob in probabilidades_validacao.items():
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    fpr, tpr, _ = roc_curve(y_validacao, prob)
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    precision, recall, _ = precision_recall_curve(y_validacao, prob)
    # Refinamos o gráfico para que a mensagem visual seja direta e elegante.
    axes[0].plot(fpr, tpr, linewidth=2, label=f"{nome} — AUC {roc_auc_score(y_validacao, prob):.3f}")
    # Refinamos o gráfico para que a mensagem visual seja direta e elegante.
    axes[1].plot(recall, precision, linewidth=2, label=f"{nome} — AP {average_precision_score(y_validacao, prob):.3f}")

# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0].plot([0, 1], [0, 1], "--", color="#AAB7BE")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0].set(title="Curva ROC — capacidade de ordenação", xlabel="Taxa de falso positivo", ylabel="Taxa de verdadeiro positivo")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1].axhline(y_validacao.mean(), linestyle="--", color="#AAB7BE", label="Prevalência")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1].set(title="Curva precisão-revocação — foco no evento raro", xlabel="Sensibilidade", ylabel="Precisão")
# Percorremos cada elemento para aplicar a mesma regra de forma consistente.
for ax in axes:
    # Refinamos o gráfico para que a mensagem visual seja direta e elegante.
    ax.legend(frameon=False)
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.tight_layout()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()

# Abrimos o portão de qualidade que decidirá se esta etapa está madura para avançar.
portao_qualidade("Etapa 4 — Modelagem e seleção", {
    # Registramos `Os conjuntos não ficaram separados` para manter esta informação nomeada e auditável.
    "Os conjuntos não ficaram separados": not (
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        (set(X_treino.index) & set(X_validacao.index))
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        or (set(X_treino.index) & set(X_teste.index))
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        or (set(X_validacao.index) & set(X_teste.index))
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    ),
    # Registramos `Menos de dois modelos foram comparados` para manter esta informação nomeada e auditável.
    "Menos de dois modelos foram comparados": len(comparacao_modelos) >= 2,
    # Registramos `A ROC AUC não foi calculada` para manter esta informação nomeada e auditável.
    "A ROC AUC não foi calculada": comparacao_modelos["roc_auc"].between(0, 1).all(),
    # Registramos `A PR AUC não foi calculada` para manter esta informação nomeada e auditável.
    "A PR AUC não foi calculada": comparacao_modelos["pr_auc"].between(0, 1).all(),
    # Registramos `O modelo vencedor não foi definido` para manter esta informação nomeada e auditável.
    "O modelo vencedor não foi definido": nome_vencedor in modelos,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})


## 5. Do score à política de crédito

Um score ordena risco; uma política decide o que fazer. O limiar abaixo minimiza um custo ilustrativo em que deixar passar um inadimplente custa cinco vezes mais que encaminhar um bom cliente para revisão. Isso **não é uma recomendação financeira pronta para produção**: a razão de custos precisa incorporar margem, recuperação, apetite de risco e capacidade operacional reais.


In [ ]:
# Criamos a função `escolher_limiar` para transformar esta ideia em uma etapa reutilizável.
def escolher_limiar(y_real, probabilidades, custo_fp=1, custo_fn=5):
    # Guardamos em `candidatos` a evidência produzida por esta operação.
    candidatos = np.linspace(0.01, 0.60, 240)
    # Guardamos em `linhas` a evidência produzida por esta operação.
    linhas = []
    # Percorremos cada elemento para aplicar a mesma regra de forma consistente.
    for limiar in candidatos:
        # Guardamos em `previsto` a evidência produzida por esta operação.
        previsto = probabilidades >= limiar
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        tn, fp, fn, tp = confusion_matrix(y_real, previsto, labels=[0, 1]).ravel()
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        linhas.append({
            # Registramos `limiar` para manter esta informação nomeada e auditável.
            "limiar": limiar,
            # Registramos `custo` para manter esta informação nomeada e auditável.
            "custo": fp * custo_fp + fn * custo_fn,
            # Registramos `fp` para manter esta informação nomeada e auditável.
            "fp": fp,
            # Registramos `fn` para manter esta informação nomeada e auditável.
            "fn": fn,
            # Registramos `tp` para manter esta informação nomeada e auditável.
            "tp": tp,
            # Registramos `tn` para manter esta informação nomeada e auditável.
            "tn": tn,
            # Registramos `sensibilidade` para manter esta informação nomeada e auditável.
            "sensibilidade": tp / (tp + fn) if tp + fn else 0,
            # Registramos `taxa_sinalizacao` para manter esta informação nomeada e auditável.
            "taxa_sinalizacao": previsto.mean(),
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        })
    # Guardamos em `tabela` a evidência produzida por esta operação.
    tabela = pd.DataFrame(linhas)
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return float(tabela.loc[tabela["custo"].idxmin(), "limiar"]), tabela

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
limiar_otimo, curva_custo = escolher_limiar(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    y_validacao, prob_validacao, CUSTO_FALSO_POSITIVO, CUSTO_FALSO_NEGATIVO
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, ax = plt.subplots(figsize=(9.5, 4.5))
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.plot(curva_custo["limiar"], curva_custo["custo"], color=CORES["azul"], linewidth=2.4)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.axvline(limiar_otimo, color=CORES["coral"], linestyle="--", label=f"Limiar escolhido: {limiar_otimo:.3f}")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set(title="O limiar traduz a função de custo em ação", xlabel="Limiar de probabilidade", ylabel="Custo relativo na validação")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.legend(frameon=False)
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()


In [ ]:
# Pedimos ao modelo probabilidades de risco, preservando a riqueza do score contínuo.
prob_teste = modelo_vencedor.predict_proba(X_teste)[:, 1]
# Guardamos em `metricas_teste` a evidência produzida por esta operação.
metricas_teste = calcular_metricas(y_teste, prob_teste, limiar_otimo)
# Guardamos em `pred_teste` a evidência produzida por esta operação.
pred_teste = (prob_teste >= limiar_otimo).astype(int)
# Guardamos em `matriz` a evidência produzida por esta operação.
matriz = confusion_matrix(y_teste, pred_teste, labels=[0, 1])

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.heatmap(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    matriz,
    # Guardamos em `annot` a evidência produzida por esta operação.
    annot=True,
    # Guardamos em `fmt` a evidência produzida por esta operação.
    fmt=",d",
    # Guardamos em `cmap` a evidência produzida por esta operação.
    cmap=sns.light_palette(CORES["turquesa"], as_cmap=True),
    # Guardamos em `cbar` a evidência produzida por esta operação.
    cbar=False,
    # Refinamos o gráfico para que a mensagem visual seja direta e elegante.
    ax=axes[0],
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0].set(title=f"Matriz de confusão — limiar {limiar_otimo:.3f}", xlabel="Predito", ylabel="Real")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0].set_xticklabels(["Sem dificuldade", "Com dificuldade"])
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[0].set_yticklabels(["Sem dificuldade", "Com dificuldade"], rotation=0)

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
fpr_t, tpr_t, _ = roc_curve(y_teste, prob_teste)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1].plot(fpr_t, tpr_t, color=CORES["coral"], linewidth=2.5)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1].plot([0, 1], [0, 1], "--", color="#AAB7BE")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[1].set(title=f"Teste interno — ROC AUC {metricas_teste['roc_auc']:.3f}", xlabel="Falso positivo", ylabel="Verdadeiro positivo")

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
fracao_positivos, probabilidade_media = calibration_curve(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    y_teste, prob_teste, n_bins=10, strategy="quantile"
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[2].plot(probabilidade_media, fracao_positivos, marker="o", color=CORES["dourado"], linewidth=2.2)
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[2].plot([0, 1], [0, 1], "--", color="#AAB7BE")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
axes[2].set(
    # Guardamos em `title` a evidência produzida por esta operação.
    title=f"Calibração — Brier {metricas_teste['brier']:.3f}",
    # Guardamos em `xlabel` a evidência produzida por esta operação.
    xlabel="Probabilidade média prevista",
    # Guardamos em `ylabel` a evidência produzida por esta operação.
    ylabel="Frequência observada",
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.tight_layout()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(pd.DataFrame([metricas_teste]).style.format("{:.4f}").hide(axis="index"))


In [ ]:
# Guardamos em `avaliacao_clientes` a evidência produzida por esta operação.
avaliacao_clientes = pd.DataFrame({"target": y_teste.to_numpy(), "probabilidade": prob_teste})
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
avaliacao_clientes["decil_risco"] = pd.qcut(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    avaliacao_clientes["probabilidade"].rank(method="first"),
    # Guardamos em `q` a evidência produzida por esta operação.
    q=10,
    # Guardamos em `labels` a evidência produzida por esta operação.
    labels=list(range(1, 11)),
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Guardamos em `tabela_decis` a evidência produzida por esta operação.
tabela_decis = (
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    avaliacao_clientes.groupby("decil_risco", observed=True)
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .agg(
        # Guardamos em `clientes` a evidência produzida por esta operação.
        clientes=("target", "size"),
        # Guardamos em `inadimplentes` a evidência produzida por esta operação.
        inadimplentes=("target", "sum"),
        # Guardamos em `taxa_inadimplencia` a evidência produzida por esta operação.
        taxa_inadimplencia=("target", "mean"),
        # Guardamos em `probabilidade_media` a evidência produzida por esta operação.
        probabilidade_media=("probabilidade", "mean"),
    # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
    )
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .sort_index(ascending=False)
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .reset_index()
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
tabela_decis["lift"] = tabela_decis["taxa_inadimplencia"] / y_teste.mean()

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    tabela_decis.style
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .format({"clientes": "{:,.0f}", "inadimplentes": "{:,.0f}", "taxa_inadimplencia": "{:.2%}", "probabilidade_media": "{:.2%}", "lift": "{:.2f}x"})
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .background_gradient(subset=["taxa_inadimplencia", "lift"], cmap="OrRd")
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .hide(axis="index")
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)

# Abrimos o portão de qualidade que decidirá se esta etapa está madura para avançar.
portao_qualidade("Etapa 5 — Política e teste final", {
    # Registramos `O limiar não foi escolhido apenas na validação` para manter esta informação nomeada e auditável.
    "O limiar não foi escolhido apenas na validação": 0 < limiar_otimo < 1,
    # Registramos `A matriz de confusão não contém todo o teste` para manter esta informação nomeada e auditável.
    "A matriz de confusão não contém todo o teste": matriz.sum() == len(y_teste),
    # Registramos `O teste final não produziu ROC AUC válida` para manter esta informação nomeada e auditável.
    "O teste final não produziu ROC AUC válida": 0 <= metricas_teste["roc_auc"] <= 1,
    # Registramos `Os decis não cobrem todo o teste` para manter esta informação nomeada e auditável.
    "Os decis não cobrem todo o teste": tabela_decis["clientes"].sum() == len(y_teste),
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})


## 6. Explicabilidade, equidade e confiança

Um bom modelo de risco precisa responder três perguntas adicionais: **quais sinais pesam?**, **o desempenho muda entre grupos?** e **o que será monitorado depois?** A importância por permutação mede quanto a ROC AUC cai quando uma variável é embaralhada. Ela mostra contribuição preditiva, não causalidade.


In [ ]:
# Guardamos em `amostra_importancia` a evidência produzida por esta operação.
amostra_importancia = X_validacao.sample(min(3_000, len(X_validacao)), random_state=RANDOM_STATE)
# Guardamos em `y_importancia` a evidência produzida por esta operação.
y_importancia = y_validacao.loc[amostra_importancia.index]
# Guardamos em `perm` a evidência produzida por esta operação.
perm = permutation_importance(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    modelo_vencedor,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    amostra_importancia,
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    y_importancia,
    # Guardamos em `scoring` a evidência produzida por esta operação.
    scoring="roc_auc",
    # Guardamos em `n_repeats` a evidência produzida por esta operação.
    n_repeats=1,
    # Guardamos em `random_state` a evidência produzida por esta operação.
    random_state=RANDOM_STATE,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)
# Guardamos em `importancias` a evidência produzida por esta operação.
importancias = (
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    pd.DataFrame({"variavel": X_validacao.columns, "importancia": perm.importances_mean})
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .sort_values("importancia", ascending=False)
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .head(20)
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)

# Preparamos a visualização que dará forma ao argumento construído pelos dados.
fig, ax = plt.subplots(figsize=(9, 6.3))
# Guardamos em `plot_imp` a evidência produzida por esta operação.
plot_imp = importancias.sort_values("importancia")
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.barh(plot_imp["variavel"], plot_imp["importancia"], color=CORES["turquesa"])
# Refinamos o gráfico para que a mensagem visual seja direta e elegante.
ax.set(title="Quais variáveis sustentam a ordenação de risco?", xlabel="Queda média de ROC AUC ao permutar", ylabel="")
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
sns.despine()
# Preparamos a visualização que dará forma ao argumento construído pelos dados.
plt.show()


In [ ]:
# Criamos a função `auditar_grupo` para transformar esta ideia em uma etapa reutilizável.
def auditar_grupo(grupos, y_real, probabilidades, limiar):
    # Guardamos em `linhas` a evidência produzida por esta operação.
    linhas = []
    # Guardamos em `base` a evidência produzida por esta operação.
    base = pd.DataFrame({"grupo": grupos.astype(str), "target": y_real, "prob": probabilidades}).dropna()
    # Percorremos cada elemento para aplicar a mesma regra de forma consistente.
    for grupo, fatia in base.groupby("grupo"):
        # Guardamos em `previsto` a evidência produzida por esta operação.
        previsto = (fatia["prob"] >= limiar).astype(int)
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        tn, fp, fn, tp = confusion_matrix(fatia["target"], previsto, labels=[0, 1]).ravel()
        # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
        linhas.append({
            # Registramos `grupo` para manter esta informação nomeada e auditável.
            "grupo": grupo,
            # Registramos `clientes` para manter esta informação nomeada e auditável.
            "clientes": len(fatia),
            # Registramos `taxa_observada` para manter esta informação nomeada e auditável.
            "taxa_observada": fatia["target"].mean(),
            # Registramos `taxa_sinalizada` para manter esta informação nomeada e auditável.
            "taxa_sinalizada": previsto.mean(),
            # Registramos `sensibilidade` para manter esta informação nomeada e auditável.
            "sensibilidade": tp / (tp + fn) if tp + fn else np.nan,
            # Registramos `taxa_falso_positivo` para manter esta informação nomeada e auditável.
            "taxa_falso_positivo": fp / (fp + tn) if fp + tn else np.nan,
            # Registramos `roc_auc` para manter esta informação nomeada e auditável.
            "roc_auc": roc_auc_score(fatia["target"], fatia["prob"]) if fatia["target"].nunique() == 2 else np.nan,
        # Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
        })
    # Devolvemos o resultado construído para que a próxima etapa possa continuar a história.
    return pd.DataFrame(linhas)

# Guardamos em `genero_teste` a evidência produzida por esta operação.
genero_teste = treino_modelo.loc[X_teste.index, "CODE_GENDER"]
# Guardamos em `auditoria_genero` a evidência produzida por esta operação.
auditoria_genero = auditar_grupo(genero_teste, y_teste, prob_teste, limiar_otimo)
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    auditoria_genero.style
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .format({"clientes": "{:,.0f}", "taxa_observada": "{:.2%}", "taxa_sinalizada": "{:.2%}", "sensibilidade": "{:.2%}", "taxa_falso_positivo": "{:.2%}", "roc_auc": "{:.3f}"})
    # Encadeamos mais uma transformação, mantendo o fluxo de leitura de cima para baixo.
    .hide(axis="index")
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(Markdown(
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "**Interpretação responsável.** `CODE_GENDER` não participou do treinamento. A tabela é uma auditoria de disparidades, "
    # Incluímos este elemento na coleção que organiza os componentes da etapa.
    "não uma certificação de equidade. Antes de produção, devem ser definidos limites aceitáveis, análise jurídica, revisão humana e testes adicionais por idade e interseções de grupos."
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
))


In [ ]:
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Guardamos em `modelo_final` a evidência produzida por esta operação.
modelo_final = clone(modelo_vencedor)
# Ensinamos o modelo usando somente as observações reservadas para aprendizagem.
modelo_final.fit(X_total, y_total)

# Pedimos ao modelo probabilidades de risco, preservando a riqueza do score contínuo.
prob_kaggle = modelo_final.predict_proba(X_kaggle)[:, 1]
# Guardamos em `submissao` a evidência produzida por esta operação.
submissao = pd.DataFrame({
    # Registramos `SK_ID_CURR` para manter esta informação nomeada e auditável.
    "SK_ID_CURR": teste_kaggle["SK_ID_CURR"].astype(int),
    # Registramos `TARGET` para manter esta informação nomeada e auditável.
    "TARGET": prob_kaggle,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})

# Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
if caminho("sample_submission.csv") is not None:
    # Guardamos em `modelo_submissao` a evidência produzida por esta operação.
    modelo_submissao = pd.read_csv(caminho("sample_submission.csv"))
    # Testamos esta condição para escolher com segurança o próximo caminho da narrativa.
    if set(modelo_submissao["SK_ID_CURR"]) != set(submissao["SK_ID_CURR"]):
        # Interrompemos a história aqui, pois avançar com esta inconsistência produziria uma conclusão frágil.
        raise ValueError("Os IDs da submissão não coincidem com application_test.")
    # Guardamos em `submissao` a evidência produzida por esta operação.
    submissao = modelo_submissao[["SK_ID_CURR"]].merge(submissao, on="SK_ID_CURR", how="left", validate="one_to_one")

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
joblib.dump(modelo_final, PASTA_RESULTADOS / "modelo_risco_credito.joblib")
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
submissao.to_csv(PASTA_RESULTADOS / "submissao_home_credit.csv", index=False)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
comparacao_modelos.to_csv(PASTA_RESULTADOS / "comparacao_modelos.csv", index=False)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
tabela_decis.to_csv(PASTA_RESULTADOS / "tabela_decis.csv", index=False)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
importancias.to_csv(PASTA_RESULTADOS / "importancia_variaveis.csv", index=False)
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
auditoria_genero.to_csv(PASTA_RESULTADOS / "auditoria_genero.csv", index=False)

# Guardamos em `manifesto` a evidência produzida por esta operação.
manifesto = {
    # Registramos `modelo_selecionado` para manter esta informação nomeada e auditável.
    "modelo_selecionado": nome_vencedor,
    # Registramos `modo_execucao` para manter esta informação nomeada e auditável.
    "modo_execucao": MODO_EXECUCAO,
    # Registramos `base_completa` para manter esta informação nomeada e auditável.
    "base_completa": not faltantes,
    # Registramos `arquivos_ausentes` para manter esta informação nomeada e auditável.
    "arquivos_ausentes": faltantes,
    # Registramos `limiar_operacional_ilustrativo` para manter esta informação nomeada e auditável.
    "limiar_operacional_ilustrativo": limiar_otimo,
    # Registramos `metricas_teste` para manter esta informação nomeada e auditável.
    "metricas_teste": {k: float(v) for k, v in metricas_teste.items()},
    # Registramos `variaveis_modelo` para manter esta informação nomeada e auditável.
    "variaveis_modelo": X_total.columns.tolist(),
    # Registramos `random_state` para manter esta informação nomeada e auditável.
    "random_state": RANDOM_STATE,
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
}
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
(PASTA_RESULTADOS / "manifesto_modelo.json").write_text(
    # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
)

# Guardamos em `artefatos` a evidência produzida por esta operação.
artefatos = sorted(p.name for p in PASTA_RESULTADOS.iterdir())
# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(pd.DataFrame({"artefatos_gerados": artefatos}).style.hide(axis="index"))

# Abrimos o portão de qualidade que decidirá se esta etapa está madura para avançar.
portao_qualidade("Etapa 6 — Governança e entrega", {
    # Registramos `A auditoria por grupo não foi gerada` para manter esta informação nomeada e auditável.
    "A auditoria por grupo não foi gerada": not auditoria_genero.empty,
    # Registramos `A importância das variáveis não foi calculada` para manter esta informação nomeada e auditável.
    "A importância das variáveis não foi calculada": not importancias.empty,
    # Registramos `A submissão não cobre application_test` para manter esta informação nomeada e auditável.
    "A submissão não cobre application_test": len(submissao) == len(teste_kaggle),
    # Registramos `Há probabilidades fora de [0, 1]` para manter esta informação nomeada e auditável.
    "Há probabilidades fora de [0, 1]": submissao["TARGET"].between(0, 1).all(),
    # Registramos `O modelo serializado não foi salvo` para manter esta informação nomeada e auditável.
    "O modelo serializado não foi salvo": (PASTA_RESULTADOS / "modelo_risco_credito.joblib").exists(),
    # Registramos `A limitação de arquivos ausentes não foi registrada` para manter esta informação nomeada e auditável.
    "A limitação de arquivos ausentes não foi registrada": (not faltantes) or (manifesto["arquivos_ausentes"] == faltantes),
# Fechamos a estrutura iniciada nas linhas anteriores e consolidamos seu significado.
})


## 7. Fechamento executivo

O projeto não termina em “aprovar ou negar”. Ele entrega uma forma auditável de **ordenar risco**, transforma o histórico em sinais compreensíveis e explicita o custo da decisão. O modelo é um componente de apoio: não substitui política de crédito, validação independente, análise de impacto ou revisão humana.


In [ ]:
# Guardamos em `decil_mais_arriscado` a evidência produzida por esta operação.
decil_mais_arriscado = tabela_decis.iloc[0]
# Guardamos em `decil_menos_arriscado` a evidência produzida por esta operação.
decil_menos_arriscado = tabela_decis.iloc[-1]

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(HTML(f"""
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:12px 0 20px">
  # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
  <div style="background:#0B1F33;color:white;padding:18px;border-radius:12px"><small>ROC AUC — teste</small><br><b style="font-size:28px">{metricas_teste['roc_auc']:.3f}</b></div>
  # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
  <div style="background:#0E4D64;color:white;padding:18px;border-radius:12px"><small>PR AUC — teste</small><br><b style="font-size:28px">{metricas_teste['pr_auc']:.3f}</b></div>
  # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
  <div style="background:#137C8B;color:white;padding:18px;border-radius:12px"><small>KS — teste</small><br><b style="font-size:28px">{metricas_teste['ks']:.3f}</b></div>
  # Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
  <div style="background:#FF6B5E;color:white;padding:18px;border-radius:12px"><small>Lift — decil de maior risco</small><br><b style="font-size:28px">{decil_mais_arriscado['lift']:.2f}x</b></div>
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
</div>
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
"""))

# Apresentamos o resultado em formato visual para transformar cálculo em comunicação.
display(Markdown(f"""
### A história em quatro conclusões

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
1. **O problema é desbalanceado:** apenas {taxa_inadimplencia:.2%} dos clientes apresentam dificuldade; por isso, acurácia isolada não serve.
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
2. **O histórico é parte da identidade de risco:** {atributos_historicos} atributos resumem exposição, atraso, utilização e recência sem usar o alvo.
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
3. **O score organiza a fila:** no teste interno, o modelo alcançou ROC AUC de **{metricas_teste['roc_auc']:.3f}** e KS de **{metricas_teste['ks']:.3f}**.
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
4. **A decisão exige governança:** o limiar **{limiar_otimo:.3f}** é apenas uma simulação econômica; sensibilidade, falso positivo, equidade e estabilidade precisam de limites institucionais.

### Próximas ações antes de produção

# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
- validar o custo financeiro real e recalibrar o limiar;
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
- fazer validação temporal e fora da amostra, indisponível neste recorte;
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
- testar estabilidade populacional (PSI), calibração e deriva mensal;
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
- ampliar a auditoria de equidade para idade e interseções, com revisão jurídica;
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
- documentar versão dos dados, aprovações e estratégia de intervenção humana.
# Damos continuidade a esta etapa com a operação necessária para sustentar a conclusão seguinte.
"""))


---

### Checklist final — “dá para melhorar?”

| Etapa | Resposta para avançar | Evidência |
|---|---|---|
| Integridade dos dados | **Não** | chaves, alvo, sobreposição e completude testados |
| Análise exploratória | **Não** | desbalanceamento, ausências, anomalias e sinais comparados |
| Engenharia de atributos | **Não** | uma linha por cliente, merges validados e ausência de alvo |
| Modelagem | **Não** | treino/validação/teste separados e dois modelos comparados |
| Política | **Não** | limiar escolhido na validação e avaliado no teste intocado |
| Governança | **Não** | explicabilidade, auditoria de grupo, manifesto e artefatos salvos |

Se qualquer condição deixar de ser verdadeira em uma nova execução, o respectivo portão interrompe o notebook com **“Dá para melhorar? Sim”**.
